# 06 — Security, Anomaly Detection & Adversarial Robustness (T-GAE)

### Academic Research Objectives:
1. **Unsupervised Anomaly Detection:** Train a Temporal Graph Autoencoder (T-GAE) to detect corrupted or manipulated market feeds via Frobenius reconstruction error without invented labels.
2. **Adversarial Perturbation Experiments:** Subject dynamic graph models to controlled perturbations:
   - Feature perturbation (FGSM continuous price/volume perturbations).
   - Structural edge perturbation (random and targeted edge addition/deletion).
3. **Robustness Degradation Curves:** Evaluate predictive performance decay (AUPRC degradation) as perturbation budget $\epsilon$ increases.


In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from cryptograph.models.tgae import TemporalGAE, TemporalGraphAE


c:\Users\daksh\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Train Temporal Graph Autoencoder (T-GAE) on Normal Trailing Regimes


In [2]:
anom_file = Path('../reports/tgae_anomalies.csv')
if anom_file.exists():
    df_anom = pd.read_csv(anom_file)
    print('=== Spatio-Temporal Graph Autoencoder (T-GAE) Reconstruction Errors ===')
    print(df_anom[['event_name', 'event_date', 'baseline_reconstruction_error', 'event_reconstruction_error', 'anomaly_ratio', 'most_anomalous_asset', 'is_statistical_anomaly']].to_string())
else:
    print('reports/tgae_anomalies.csv not found.')


=== Spatio-Temporal Graph Autoencoder (T-GAE) Reconstruction Errors ===
                   event_name  event_date  baseline_reconstruction_error  event_reconstruction_error  anomaly_ratio most_anomalous_asset  is_statistical_anomaly
0    US spot BTC ETF approval  2024-01-10                       0.024117                    0.022693       0.940936                   OP                   False
1      Yen carry-trade unwind  2024-08-05                       0.024117                    0.062353       2.585407                 AAVE                   False
2  Oct-10 liquidation cascade  2025-10-10                       0.024117                    0.006933       0.287484                  FET                   False


## 2. Evaluate Reconstruction Error Spikes During Market Anomalies


In [3]:
if anom_file.exists():
    fig = px.bar(
        df_anom,
        x='event_name',
        y='anomaly_ratio',
        color='is_statistical_anomaly',
        title='T-GAE Reconstruction Error Ratio: Market Stress Events vs. Baseline',
        labels={'anomaly_ratio': 'Reconstruction Error Ratio (Event / Baseline)', 'event_name': 'Market Event'}
    )
    fig.add_hline(y=1.5, line_dash='dash', line_color='red', annotation_text='Anomaly Threshold (1.5x)')
    fig.update_layout(template='plotly_white')
    fig.show()


## 3. Adversarial Feature & Structural Perturbation Degradation Curves


In [4]:
deg_file = Path('../reports/robustness_degradation.csv')
if deg_file.exists():
    df_deg = pd.read_csv(deg_file)
    print('=== Perturbation Degradation Curves ===')
    print(df_deg.to_string())
    
    fig = px.line(
        df_deg,
        x='epsilon',
        y=['prediction_drift_mse', 'explanation_jaccard_stability'],
        markers=True,
        title='Adversarial Resilience: Prediction Drift & Explanation Stability vs. Perturbation Budget',
        labels={'epsilon': 'Perturbation Budget (Epsilon)', 'value': 'Metric Score'}
    )
    fig.update_layout(template='plotly_white')
    fig.show()
else:
    print('reports/robustness_degradation.csv not found.')


=== Perturbation Degradation Curves ===
               perturbation_type  epsilon  prediction_drift_mse  explanation_jaccard_stability
0  random_gaussian_and_edge_drop     0.01              0.000109                       0.250000
1  random_gaussian_and_edge_drop     0.05              0.000410                       0.428571
2  random_gaussian_and_edge_drop     0.10              0.012020                       0.250000
3  random_gaussian_and_edge_drop     0.20              0.985221                       0.428571
4  random_gaussian_and_edge_drop     0.35              0.081198                       0.250000
5               adversarial_fgsm     0.01              0.001990                       0.250000
6               adversarial_fgsm     0.05              0.044942                       0.428571
7               adversarial_fgsm     0.10              0.175292                       0.428571
8               adversarial_fgsm     0.20              0.707491                       0.250000
9         